In [ ]:
Install packages and mount the google drive

In [ ]:
!pip install pyspark -q

from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs("/content/drive/MyDrive/chicago-crime-ml/data", exist_ok=True)
os.makedirs("/content/drive/MyDrive/chicago-crime-ml/outputs", exist_ok=True)
print("Google Drive mounted and folders ready")

In [ ]:
Pull from bigquery 

In [ ]:
from google.colab import auth
auth.authenticate_user()

from google.cloud import bigquery
import pandas as pd

PROJECT_ID = "healthy-result-491819-e4"
client = bigquery.Client(project=PROJECT_ID)

QUERY = """
SELECT
    primary_type,
    description,
    location_description,
    arrest,
    domestic,
    district,
    ward,
    community_area,
    EXTRACT(YEAR FROM date)         AS year,
    EXTRACT(MONTH FROM date)        AS month,
    EXTRACT(DAYOFWEEK FROM date)    AS day_of_week,
    EXTRACT(HOUR FROM date)         AS hour_of_day,
    latitude,
    longitude
FROM `bigquery-public-data.chicago_crime.crime`
WHERE date IS NOT NULL
    AND primary_type IS NOT NULL
    AND district IS NOT NULL
    AND EXTRACT(YEAR FROM date) BETWEEN 2015 AND 2023
"""

print("Pulling data from BigQuery...")
df_raw = client.query(QUERY).to_dataframe()
print(f"Loaded {len(df_raw):,} rows")
print(f"\nCrime type distribution (top 10):")
print(df_raw["primary_type"].value_counts().head(10))

df_raw.to_csv("/content/drive/MyDrive/chicago-crime-ml/data/chicago_crime_raw.csv", index=False)
print("\nSaved raw data to Google Drive")

In [ ]:
PySpark Preprocessing

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType
from pyspark.ml.feature import StringIndexer

spark = SparkSession.builder \
    .appName("ChicagoCrimePreprocessing") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("Spark session started")

data = spark.read.csv(
    "/content/drive/MyDrive/chicago-crime-ml/data/chicago_crime_raw.csv",
    header=True,
    inferSchema=True
)
print(f"Loaded {data.count():,} rows, {len(data.columns)} columns")

# drop nulls: now including ward, community_area, description
key_cols = [
    "primary_type", "district", "ward", "community_area",
    "year", "month", "day_of_week", "hour_of_day",
    "arrest", "domestic", "location_description", "description"
]
df_clean = data.dropna(subset=key_cols)
print(f"After dropping nulls: {df_clean.count():,} rows")

# cast boolean columns to integers (0/1)
df_clean = df_clean \
    .withColumn("arrest_int",   F.col("arrest").cast(IntegerType())) \
    .withColumn("domestic_int", F.col("domestic").cast(IntegerType()))

# is it rush hour? (7-9am or 4-7pm)
df_clean = df_clean.withColumn(
    "is_rush_hour",
    F.when(
        (F.col("hour_of_day").between(7, 9)) |
        (F.col("hour_of_day").between(16, 19)),
        1
    ).otherwise(0)
)

# is it a weekend? (1=Sunday, 7=Saturday in Spark)
df_clean = df_clean.withColumn(
    "is_weekend",
    F.when(F.col("day_of_week").isin([1, 7]), 1).otherwise(0)
)

# season (1=winter, 2=spring, 3=summer, 4=fall)
df_clean = df_clean.withColumn(
    "season",
    F.when(F.col("month").isin([12, 1, 2]), 1)
     .when(F.col("month").isin([3, 4, 5]),  2)
     .when(F.col("month").isin([6, 7, 8]),  3)
     .otherwise(4)
)

# interaction feature: district × hour_of_day
# captures that crime patterns vary by district AND time together
# e.g. district 8 at 2am is very different from district 18 at 2am
df_clean = df_clean.withColumn(
    "district_hour",
    F.col("district") * 100 + F.col("hour_of_day")
)

# group description into broad categories
# this avoids leaking crime type while still capturing severity/method
df_clean = df_clean.withColumn(
    "description_grouped",
    F.when(F.lower(F.col("description")).contains("aggravated"), "AGGRAVATED")
     .when(F.lower(F.col("description")).contains("armed"),      "ARMED")
     .when(F.lower(F.col("description")).contains("attempt"),    "ATTEMPT")
     .when(F.lower(F.col("description")).contains("simple"),     "SIMPLE")
     .when(F.lower(F.col("description")).contains("domestic"),   "DOMESTIC")
     .when(F.lower(F.col("description")).contains("retail"),     "RETAIL")
     .when(F.lower(F.col("description")).contains("financial"),  "FINANCIAL")
     .when(F.lower(F.col("description")).contains("possess"),    "POSSESSION")
     .when(F.lower(F.col("description")).contains("poss"),       "POSSESSION")
     .when(F.lower(F.col("description")).contains("deliver"),    "DELIVERY")
     .when(F.lower(F.col("description")).contains("vehicle"),    "VEHICLE")
     .when(F.lower(F.col("description")).contains("damage"),     "DAMAGE")
     .when(F.lower(F.col("description")).contains("forcible"),   "FORCIBLE")
     .otherwise("OTHER")
)

print("\nDescription group distribution:")
df_clean.groupBy("description_grouped") \
    .count() \
    .orderBy(F.desc("count")) \
    .show()

# encode location_description (top 50 + OTHER)
top_locations = (
    df_clean.groupBy("location_description")
            .count()
            .orderBy(F.desc("count"))
            .limit(50)
            .select("location_description")
            .rdd.flatMap(lambda x: x)
            .collect()
)
df_clean = df_clean.withColumn(
    "location_description_clean",
    F.when(F.col("location_description").isin(top_locations),
           F.col("location_description"))
     .otherwise("OTHER")
)
location_indexer = StringIndexer(
    inputCol="location_description_clean",
    outputCol="location_encoded"
)
df_clean = location_indexer.fit(df_clean).transform(df_clean)

# encode description_grouped
desc_indexer = StringIndexer(
    inputCol="description_grouped",
    outputCol="description_encoded"
)
df_clean = desc_indexer.fit(df_clean).transform(df_clean)

# filter to top 10 crime types
top_n = 10
top_types = (
    df_clean.groupBy("primary_type")
            .count()
            .orderBy(F.desc("count"))
            .limit(top_n)
            .select("primary_type")
)
df_model1 = df_clean.join(top_types, on="primary_type", how="inner")
print(f"\nModel 1 dataset (top {top_n} crime types): {df_model1.count():,} rows")

df_model1.groupBy("primary_type") \
    .count() \
    .orderBy(F.desc("count")) \
    .show()

# encode crime type label
indexer = StringIndexer(inputCol="primary_type", outputCol="label")
df_model1 = indexer.fit(df_model1).transform(df_model1)

# save model 1: now 15 features
feature_cols = [
    "year", "month", "day_of_week", "hour_of_day",
    "district", "ward", "community_area",
    "arrest_int", "domestic_int",
    "is_rush_hour", "is_weekend", "season",
    "location_encoded",
    "description_encoded",
    "district_hour"
]
cols_to_save = feature_cols + ["label", "primary_type"]
df_model1.select(cols_to_save).toPandas() \
    .to_csv("/content/drive/MyDrive/chicago-crime-ml/data/model1_classification.csv", index=False)
print("Saved model1_classification.csv to Google Drive")

# aggregate for model 2
df_model2 = df_clean.groupBy("district", "year", "month", "season") \
    .agg(F.count("*").alias("crime_count")) \
    .orderBy("year", "month", "district")
df_model2.toPandas() \
    .to_csv("/content/drive/MyDrive/chicago-crime-ml/data/model2_regression.csv", index=False)
print("Saved model2_regression.csv to Google Drive")

print("\nPreprocessing complete!")
spark.stop()